# Predictive Analytics: Support Vector Machines with Regression for Community Areas

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

We used the GPU to train this model. In case the model shoulde be trained on the CPU. Change USE_GPU to false.

In [55]:
USE_GPU = False
from run_config import PATHS, MODELS_DIR

In [26]:
if USE_GPU:
    %load_ext cuml.accel
from run_config import PATHS

In [27]:
if USE_GPU:
    import os
    os.environ["LD_LIBRARY_PATH"] = "/mnt/c/Users/bkran/Documents/AAA/Group-3-AAA/.venv/lib64/python3.12/site-packages/nvidia/cuda_runtime/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

    import cuml
    print(cuml.__version__)

In [ ]:
TRAIN_SAMPLE = 70_000 
GRID_SAMPLE = 70_000 # if validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_UNIT = "COMMUNITY_AREAS" # COMMUNITY_AREAS
SPATIAL_ENCODING = "latlong" # options: latlong, onehot
TIME_UNIT = "1H" # options: 1H, 4H, 24H

In [29]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv" # same file in full/sample
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [30]:
import pandas as pd
import geopandas as gpd
import numpy as np
import polars as pl
from shapely import wkt

if USE_GPU:
    # cuml
    from cuml import SVR
    from cuml import LinearSVR
else:
    #sklearn
    from sklearn.svm import SVR 
    from sklearn.svm import LinearSVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.experimental import enable_halving_search_cv # noqa
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.utils import resample
from sklearn.base import clone

# joblib
from joblib import load, dump
from joblib import Memory




## Preparations

In [31]:
INPUT = PATHS.train_test_dir

In [32]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [33]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [34]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-12-22 10:00:00,12,1,10,-0.500000,8.660254e-01,0.000000,1.000000,0.500000,-8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
1,2025-08-18 04:00:00,8,1,4,-0.500000,-8.660254e-01,0.000000,1.000000,0.866025,5.000000e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
2,2025-09-19 23:00:00,9,5,23,-0.866025,-5.000000e-01,-0.433884,-0.900969,-0.258819,9.659258e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
3,2025-11-22 07:00:00,11,6,7,-0.866025,5.000000e-01,-0.974928,-0.222521,0.965926,-2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
4,2025-09-24 20:00:00,9,3,20,-0.866025,-5.000000e-01,0.974928,-0.222521,-0.866025,5.000000e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626467,2025-08-20 05:00:00,8,3,5,-0.500000,-8.660254e-01,0.974928,-0.222521,0.965926,2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
626468,2025-04-11 06:00:00,4,5,6,1.000000,6.123234e-17,-0.433884,-0.900969,1.000000,6.123234e-17,...,0.0,0.0,0.0,0.0,0.0,308.37,20.558,5.13,55.31,Mobile
626469,2025-06-06 21:00:00,6,5,21,0.500000,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
626470,2025-11-10 00:00:00,11,1,0,-0.866025,5.000000e-01,0.000000,1.000000,0.000000,1.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips


In [35]:
if len(train_df) > TRAIN_SAMPLE:
   train_df = train_df.sample(n=TRAIN_SAMPLE, random_state=40)

In [36]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
192563,2025-06-30 06:00:00,6,1,6,0.5,-8.660254e-01,0.000000,1.000000,1.000000,6.123234e-17,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
201427,2025-08-22 21:00:00,8,5,21,-0.5,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
118253,2026-04-08 15:00:00,4,3,15,1.0,6.123234e-17,0.974928,-0.222521,-0.707107,-7.071068e-01,...,6.0,1108.13,8.148015,0.0,54.0,8712.28,64.060882,16.05,172.5,Credit Card
363574,2025-12-06 23:00:00,12,6,23,-0.5,8.660254e-01,-0.974928,-0.222521,-0.258819,9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
198821,2026-02-23 11:00:00,2,1,11,0.5,8.660254e-01,0.000000,1.000000,0.258819,-9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips


In [37]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [38]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [39]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
192563,2025-06-30 06:00:00,6,1,6,0.5,-8.660254e-01,0.000000,1.000000,1.000000,6.123234e-17,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
201427,2025-08-22 21:00:00,8,5,21,-0.5,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
118253,2026-04-08 15:00:00,4,3,15,1.0,6.123234e-17,0.974928,-0.222521,-0.707107,-7.071068e-01,...,6.0,1108.13,8.148015,0.0,54.0,8712.28,64.060882,16.05,172.5,Credit Card
363574,2025-12-06 23:00:00,12,6,23,-0.5,8.660254e-01,-0.974928,-0.222521,-0.258819,9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
198821,2026-02-23 11:00:00,2,1,11,0.5,8.660254e-01,0.000000,1.000000,0.258819,-9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313745,2026-01-20 21:00:00,1,2,21,0.0,1.000000e+00,0.781831,0.623490,-0.707107,7.071068e-01,...,0.0,7.50,0.208333,0.0,1.5,346.62,9.628333,4.75,20.5,Mobile
433533,2026-02-04 18:00:00,2,3,18,0.5,8.660254e-01,0.974928,-0.222521,-1.000000,-1.836970e-16,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
77395,2025-02-17 10:00:00,2,1,10,0.5,8.660254e-01,0.000000,1.000000,0.500000,-8.660254e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
256822,2026-04-09 20:00:00,4,4,20,1.0,6.123234e-17,0.433884,-0.900969,-0.866025,5.000000e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips


In [40]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: community_area")

    census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
    census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

    census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  


    gdf_proj = gdf.to_crs(epsg=3435)
    gdf["lon"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).x
    gdf["lat"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).y

    tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

    for df in (train_df, val_df, test_df):
        df["community_area"] = df["community_area"].astype(str).str.zfill(2)
        df["lat"] = df["community_area"].map(tract_centroids["lat"])
        df["lon"] = df["community_area"].map(tract_centroids["lon"])

        # sanity check, catch silent join failures early
        n_missing_lat = df["lat"].isna().sum()
        n_missing_lon = df["lon"].isna().sum()
        if n_missing_lat or n_missing_lon:
            print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")


Encoding: latlong and Unit: community_area


In [41]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else:
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Onehot

In [42]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "COMMUNITY_AREAS"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
    # X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train
    X_test = X_test.reindex(columns=train_columns, fill_value=0)
    #X_val = X_val.reindex(columns=train_columns, fill_value=0)

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = pd.get_dummies(val_df_grid[feature_cols], columns=["community_area"])
    X_val_grid = X_val_grid.reindex(columns=train_columns, fill_value=0)

Create y

In [43]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [44]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
192563,0.5,-8.660254e-01,0.000000,1.000000,1.000000,6.123234e-17,0,1.637913,26.0,3.0,...,0,0,0,0,1,0,0,0.029036,-0.745148,0.666267
201427,-0.5,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,0,4.769125,19.0,2.0,...,1,0,0,0,1,0,0,0.029966,-0.744927,0.666473
118253,1.0,6.123234e-17,0.974928,-0.222521,-0.707107,-7.071068e-01,0,2.797738,116.0,10.0,...,0,0,0,0,0,1,0,0.027324,-0.742926,0.668815
363574,-0.5,8.660254e-01,-0.974928,-0.222521,-0.258819,9.659258e-01,0,1.637913,26.0,3.0,...,0,0,1,0,1,0,0,0.029036,-0.745148,0.666267
198821,0.5,8.660254e-01,0.000000,1.000000,0.258819,-9.659258e-01,0,1.637913,26.0,3.0,...,0,1,0,0,1,0,0,0.029036,-0.745148,0.666267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313745,0.0,1.000000e+00,0.781831,0.623490,-0.707107,7.071068e-01,0,14.645556,472.0,89.0,...,0,0,1,0,1,0,0,0.030852,-0.743918,0.667558
433533,0.5,8.660254e-01,0.974928,-0.222521,-1.000000,-1.836970e-16,0,20.086383,113.0,10.0,...,0,0,0,0,0,1,0,0.029997,-0.742829,0.668809
77395,0.5,8.660254e-01,0.000000,1.000000,0.500000,-8.660254e-01,1,13.089723,18.0,13.0,...,0,0,0,0,1,0,0,0.031294,-0.745065,0.666258
256822,1.0,6.123234e-17,0.433884,-0.900969,-0.866025,5.000000e-01,0,14.180219,413.0,35.0,...,0,1,0,0,1,0,0,0.030177,-0.743685,0.667848


### Grid Search

In [45]:
model = SVR()

In [46]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()), # only included for 24H due to small dataset
    ('svm', SVR(max_iter=50_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel # excluded C=100, C=10, 0,001 excluded via testing due to convergance issues
param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], # 4h: , 1h: 24h: 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300],  # for 24h 10, 30 and 4h, 1h: 100, 300
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300], # for 24h 10, 30 and 4h, 1h: 100, 300
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.12351894602456864 best params: {'regressor__svm__C': 30, 'regressor__svm__epsilon': 0.1}


C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficien

rbf_sigmoid best score: 0.4717793337216473 best params: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.01}


C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficien

poly best score: 0.1846166800489415 best params: {'regressor__feature_map__degree': 3, 'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 100, 'regressor__svm__C': 1, 'regressor__svm__epsilon': 0.1}
Overall best: rbf_sigmoid {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.01}


In [47]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.01}
Best CV score: 0.4717793337216473


### Train Model

In [48]:
best_model = grid_search.best_estimator_

In [49]:
# Train SVR 

best_model.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","Pipeline(memo..., tol=0.01))])"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](41,)","['month_sin','month_cos','weekday_sin',...,'x','y','z']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,41
regressor_ regressor_: objectFitted regressor.,Pipeline,"Pipeline(memo..., tol=0.01))])"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,StandardScaler,StandardScaler()
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('feature_map', ...), ...]"


In [50]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [51]:
y_pred

array([ 0.26201766,  0.15169288, -1.87274182, ..., 30.92571983,
        7.13048343, 14.08535651])

In [52]:
# Evaluation metrics
result = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2 Score": r2_score(y_test, y_pred),
}

In [56]:
df = pd.DataFrame({ # did not reorder at any point
    "y_pred": y_pred,
    "y_test": y_test,
    "community_area": test_df["community_area"].values,
    "date": test_df["datetime_hour"].values,
})
df.to_csv( MODELS_DIR / f"svm/model_{SPATIAL_UNIT}_{TIME_UNIT}.csv", index=False)
pd.DataFrame([result]).to_csv(MODELS_DIR / f"svm/result_{SPATIAL_UNIT}_{TIME_UNIT}.csv", index=False)

In [57]:
# save model
dump(best_model, MODELS_DIR / f"svm/model_{SPATIAL_UNIT}_{TIME_UNIT}_svr.joblib")
dump(grid_search, MODELS_DIR / f"svm/grid_{SPATIAL_UNIT}_{TIME_UNIT}_svr.joblib")

['C:\\Users\\bkran\\Documents\\AAA\\Group-3-AAA\\models\\full\\svm\\grid_COMMUNITY_AREAS_1H_svr.joblib']